# Cyberbullying Detection - Research Notebook

## 1. Setup and Imports

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Get the project root directory (parent of notebooks folder)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))

# Try multiple possible data locations
possible_paths = [
    os.path.join(PROJECT_ROOT, 'data'),
    os.path.join(PROJECT_ROOT, 'datasets'),
    os.path.join(PROJECT_ROOT, 'archive'),
    '/home/akarsh/Documents/archive',  # Original path
    'C:\\data\\archive',  # Windows possible path
    'D:\\data\\archive',  # Windows possible path
]

# Find the actual data path
DATA_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        DATA_PATH = path
        break

# If no data found, check if data folder exists in project
if DATA_PATH is None:
    # Check for data in common project locations
    project_data = os.path.join(os.getcwd(), '..', 'data')
    if os.path.exists(project_data):
        DATA_PATH = project_data

OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'output')

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Path: {DATA_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print("\nLibraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load all datasets
datasets = {}

csv_files = [
    'twitter_parsed_dataset.csv',
    'twitter_racism_parsed_dataset.csv',
    'twitter_sexism_parsed_dataset.csv',
    'aggression_parsed_dataset.csv',
    'attack_parsed_dataset.csv',
    'toxicity_parsed_dataset.csv',
    'kaggle_parsed_dataset.csv',
    'youtube_parsed_dataset.csv'
]

if DATA_PATH is None:
    print("WARNING: Data path not found!")
    print("Please ensure datasets are available in one of these locations:")
    for path in possible_paths:
        print(f"  - {path}")
    print("\nCreating sample data for demonstration...")
    
    # Create sample data for testing
    sample_data = {
        'Text': [
            'You are amazing! Great work!',
            'Hey, are you doing okay?',
            'I love spending time with you guys!',
            'This is terrible and stupid',
            'You are such an idiot!',
            'Great job on the project',
            'Nice work friend!',
            'What a horrible person you are',
        ],
        'oh_label': [0, 0, 0, 1, 1, 0, 0, 1],
        'dse_label': [0, 0, 0, 1, 1, 0, 0, 1],
    }
    twitter_df = pd.DataFrame(sample_data)
    datasets['twitter'] = twitter_df
    datasets['sample'] = twitter_df
    print(f"Created sample dataset with {len(twitter_df)} rows for demonstration")
else:
    for file in csv_files:
        try:
            file_path = os.path.join(DATA_PATH, file)
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                datasets[file.replace('_parsed_dataset.csv', '')] = df
                print(f"Loaded {file}: {len(df)} rows")
            else:
                print(f"File not found: {file_path}")
        except Exception as e:
            print(f"Error loading {file}: {e}")

print(f"\nTotal datasets loaded: {len(datasets)}")

## 3. Explore Dataset Structure

In [ ]:
# Explore each dataset
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    display(df.head(2))

## 4. Analyze Label Distribution

In [ ]:
# Analyze label columns in each dataset
for name, df in datasets.items():
    # Find label columns
    label_cols = [col for col in df.columns if 'label' in col.lower()]
    
    if label_cols:
        print(f"\n{name}: {label_cols}")
        for col in label_cols:
            print(f"  {col}:")
            print(f"    Unique values: {df[col].nunique()}")
            print(f"    Value counts:\n{df[col].value_counts().head()}")

## 5. Data Preprocessing

In [ ]:
# Combine datasets with common format
# Focus on twitter dataset first

twitter_df = datasets.get('twitter', None)

if twitter_df is not None:
    print("Twitter dataset columns:", twitter_df.columns.tolist())
    print("\nSample data:")
    display(twitter_df.head())
else:
    # Use sample data if available
    for name, df in datasets.items():
        if 'Text' in df.columns:
            twitter_df = df
            print(f"Using {name} dataset as main dataset")
            print("Columns:", twitter_df.columns.tolist())
            display(twitter_df.head())
            break

## 6. Text Statistics

In [ ]:
# Calculate text statistics
text_column = None
for col in ['Text', 'text', 'tweet', 'content']:
    if col in twitter_df.columns:
        text_column = col
        break

if text_column:
    twitter_df['text_length'] = twitter_df[text_column].astype(str).apply(len)
    twitter_df['word_count'] = twitter_df[text_column].astype(str).apply(lambda x: len(x.split()))
    
    print("Text Length Statistics:")
    print(twitter_df['text_length'].describe())
    
    print("\nWord Count Statistics:")
    print(twitter_df['word_count'].describe())
else:
    print("No text column found in dataset")

## 7. Visualize Data

In [ ]:
# Plot text length distribution
if 'text_length' in twitter_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Text length distribution
    axes[0].hist(twitter_df['text_length'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Text Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Text Length')
    
    # Word count distribution
    axes[1].hist(twitter_df['word_count'], bins=50, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Word Count')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'text_statistics.png'), dpi=150)
    plt.show()
    print(f"Saved to: {os.path.join(OUTPUT_PATH, 'text_statistics.png')}")

## 8. Label Distribution

In [ ]:
# Plot label distribution
label_column = None
for col in ['oh_label', 'dse_label', 'label', 'class', 'target']:
    if col in twitter_df.columns:
        label_column = col
        break

if label_column:
    label_counts = twitter_df[label_column].value_counts()
    
    plt.figure(figsize=(8, 5))
    label_counts.plot(kind='bar', edgecolor='black')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.title(f'Label Distribution ({label_column})')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'label_distribution.png'), dpi=150)
    plt.show()
    
    print("\nLabel Distribution:")
    print(label_counts)
else:
    print("No label column found in dataset")

## 9. Prepare Data for Model Training

In [ ]:
# Create combined dataset for training
# We'll focus on binary classification: bullying vs not_bullying

def prepare_combined_dataset(datasets):
    """Combine multiple datasets into one."""
    combined = []
    
    for name, df in datasets.items():
        # Check for common columns
        text_col = None
        for col in ['Text', 'text', 'tweet', 'content']:
            if col in df.columns:
                text_col = col
                break
        
        label_col = None
        for col in ['oh_label', 'dse_label', 'label', 'class', 'target']:
            if col in df.columns:
                label_col = col
                break
        
        if text_col and label_col:
            temp_df = df[[text_col, label_col]].copy()
            temp_df.columns = ['Text', 'label']  # Standardize column names
            temp_df['source'] = name
            combined.append(temp_df)
    
    return pd.concat(combined, ignore_index=True)

combined_df = prepare_combined_dataset(datasets)
print(f"Combined dataset shape: {combined_df.shape}")
print(f"\nLabel distribution:")
print(combined_df['label'].value_counts())

## 10. Next Steps - Model Training

In [ ]:
# Save processed data
combined_df.to_csv(os.path.join(OUTPUT_PATH, 'combined_dataset.csv'), index=False)

# Summary statistics
print("=" * 60)
print("RESEARCH SUMMARY")
print("=" * 60)
print(f"Total samples: {len(combined_df)}")
print(f"Columns: {combined_df.columns.tolist()}")
print(f"\nLabel distribution:")
print(combined_df['label'].value_counts())
print(f"\nData saved to: {OUTPUT_PATH}")